In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = r'InflAdj_datasets\InflAdj_Data_90th_2019_2025.csv'

def sector_level_heatmap(save_path=None):
    df = pd.read_csv(CSV_PATH, parse_dates=['report_date'])
    df = df[df['symbol'] != 'RNWF']
    df = df.dropna(subset=['industry', 'report_date', 'close'])
    df['report_date'] = pd.to_datetime(df['report_date'])

    sector_daily = (
        df.groupby(['industry', pd.Grouper(key='report_date', freq='B')])['close']
        .mean()
    )

    sector_daily = sector_daily.reset_index(drop=False).merge(
        sector_daily.reset_index(drop=False)[['report_date', 'industry']]
        .groupby('industry').min().reset_index(),
        on='industry', suffixes=('', '_min')
    ).sort_values(['report_date_min', 'industry']).set_index(['industry', 'report_date'])

    original_order = sector_daily.index.get_level_values('industry').unique()
    df_rotated = sector_daily['close'].unstack('report_date').reindex(original_order)
    sector_daily = df_rotated

    sector_mom = sector_daily.pct_change(periods=30, axis=1) * 100
    sector_mom = sector_mom.dropna(how='all', axis=1)

    n_sectors, n_mo = sector_mom.shape
    dates = sector_mom.columns

    finite = sector_mom.values[np.isfinite(sector_mom.values)]
    vlim = round(float(np.percentile(np.abs(finite), 95)), 1) if len(finite) > 0 else 10.0

    # ── layout ──────────────────────────────────────────────────────────────
    cell_h   = 0.52
    fig_w    = max(18, n_mo * 0.035 + 4)
    fig_h    = max(7, n_sectors * cell_h + 3.6)   # +0.4 for extra header room
    header_h = 1.5                                  # fixed inches — enough for any fig_h
    footer_h = 0.45
    T = 1.0 - header_h / fig_h
    B = footer_h / fig_h

    FACE = '#F5F4EF'
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor=FACE)
    fig.subplots_adjust(top=T, bottom=B, left=0.18, right=0.985)
    ax.set_facecolor(FACE)

    # ── colormap (unchanged) ─────────────────────────────────────────────────
    red, grey, green = '#D06A4C', FACE, '#4A9090'
    cmap = mcolors.LinearSegmentedColormap.from_list('RdGrGn', [red, grey, green])

    im = ax.imshow(
        sector_mom.values,
        aspect='auto', cmap=cmap,
        vmin=-vlim, vmax=vlim,
        interpolation='none',
    )

    # ── y-axis ───────────────────────────────────────────────────────────────
    ax.set_yticks(range(n_sectors))
    ax.set_yticklabels(sector_mom.index, fontsize=10, va='center',
                       fontfamily='monospace')
    ax.tick_params(axis='y', length=0, pad=6)

    # ── x-axis (top) ─────────────────────────────────────────────────────────
    step = max(1, n_mo // 14)
    tick_pos = list(range(0, n_mo, step))
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(
        [dates[i].strftime('%b %y') for i in tick_pos],
        fontsize=8, ha='center', rotation=0, color='#444444'
    )
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ax.tick_params(axis='x', length=0, pad=3)

    # ── thin row separators ───────────────────────────────────────────────────
    for i in range(1, n_sectors):
        ax.axhline(i - 0.5, color=FACE, linewidth=1.2, zorder=3)

    # ── spines off ───────────────────────────────────────────────────────────
    for sp in ax.spines.values():
        sp.set_visible(False)

    # ── header (all elements expressed in absolute inches above T) ────────────
    # yin converts inches-above-T → figure fraction (always > T, never touches heatmap)
    def yin(inches): return T + inches / fig_h

    L = 0.18   # left margin (matches subplots_adjust left)

    # title: 1.2 in above T
    fig.text(L, yin(1.20), 'Sector Heatmap',
             fontsize=16, fontweight='bold', va='bottom', ha='left', color='#1a1a1a')

    # subtitle: 0.82 in above T
    date_range = f"{dates[0].strftime('%b %Y')} – {dates[-1].strftime('%b %Y')}"
    fig.text(L, yin(0.82), f'1-month % change in avg sector close price  ·  {date_range}',
             fontsize=9, va='bottom', ha='left', color='#666666')

    # colorbar: 0.55 in above T, anchored to the RIGHT side of the figure
    # cbar_bottom = yin(0.55)
    # cbar_h_frac = 0.20 / fig_h
    # cbar_ax = fig.add_axes([0.63, cbar_bottom, 0.32, cbar_h_frac])
    # cb = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    # cb.set_ticks([-vlim, 0, vlim])
    # cb.set_ticklabels([f'−{vlim:.0f}%', '0', f'+{vlim:.0f}%'])
    # cb.ax.tick_params(labelsize=7.5, length=0)
    # cb.outline.set_visible(False)

    # ── footer ───────────────────────────────────────────────────────────────
    fig.text(L, 0.008,
             f'Source: InflAdj_Data_90th_2019_2025.csv  ·  {n_sectors} sectors  ·  30 trading-day rolling return',
             fontsize=7.5, color='#aaaaaa', va='bottom', ha='left')

    out = save_path or 'heatmap_sector_averages.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=FACE)
    print(f'Saved → {out}')
    plt.show()
    return fig


sector_level_heatmap()
